# Tab-DDPM Hybrid Synthetic Transaction Generator (Colab Pro + A100)

**Fraud Detection Framework -- Uncharted Data Challenge**

This notebook uses a **hybrid approach**:
- **Gaussian diffusion** (Tab-DDPM) for numerical columns (amounts, ages, hours, etc.)
- **Profile-weighted sampling** for categorical columns (fraud vectors, languages, instruments)

This avoids multinomial diffusion mode-collapse where Tab-DDPM collapses to one dominant category on small datasets.

**Runtime:** Set to **GPU (A100)** via `Runtime > Change runtime type > A100`

| Setting | Value |
|---------|-------|
| Epochs | 700 |
| Samples per archetype | 5,000 |
| Diffusion timesteps | 500 |
| Numericals | Gaussian diffusion |
| Categoricals | Profile-weighted sampling |
| Archetypes | remittance, gig_worker, unbanked, itin |

## 1. Setup & GPU Check

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    DEVICE = "cuda"
else:
    print("WARNING: No GPU detected. Set Runtime > Change runtime type > GPU")
    DEVICE = "cpu"
print(f"\nUsing device: {DEVICE}")

In [ ]:
# Clone tab-ddpm library
!git clone --quiet https://github.com/rotot0/tab-ddpm.git /content/tab-ddpm
import sys
sys.path.insert(0, "/content/tab-ddpm")

from tab_ddpm import GaussianMultinomialDiffusion
from tab_ddpm.modules import MLPDiffusion
print("tab-ddpm loaded successfully")

In [ ]:
import numpy as np
import pandas as pd
import uuid
import json
import os
from datetime import datetime, timezone, timedelta

# Create output directories
ARCHETYPES = ["remittance", "gig_worker", "unbanked", "itin"]
for arch in ARCHETYPES:
    os.makedirs(f"/content/output/{arch}", exist_ok=True)
print("Imports and output dirs ready")

## 2. Profile Configs (behavioral distributions from 1,040 scraped records)

In [ ]:
PROFILES = {
    "remittance": {
        "archetype": "remittance",
        "description": "Cross-border money transfer fraud targeting immigrant communities",
        "demographics": {"age_range": (25, 65), "income_bracket": "low-to-middle", "banking_status": "banked or semi-banked",
            "primary_communities": ["Mexican diaspora", "Filipino diaspora", "Nigerian diaspora", "Ghanaian diaspora", "Haitian diaspora", "Indian diaspora"]},
        "language_mix": {"en": 0.78, "es": 0.06, "vi": 0.07, "yo": 0.05, "hi": 0.02, "ht": 0.02},
        "fraud_vectors": {"wire transfer": 0.21, "exchange rate": 0.15, "emergency": 0.15, "interception": 0.08,
            "fake family": 0.07, "estafa": 0.06, "fraude": 0.06, "bonus": 0.05, "Western Union": 0.06, "MoneyGram": 0.04, "unknown": 0.07},
        "instruments": ["Western Union", "MoneyGram", "Remitly", "Xoom", "Wire transfer (bank)", "Hawala", "Cash pickup"],
        "transaction_patterns": {"amount_range_usd": (50, 5000), "median_amount_usd": 500,
            "peak_days": ["Friday", "Saturday"], "peak_hours": (17, 22), "typical_fee_pct": (2.0, 8.0)},
        "temporal": {"seasonal_peaks": ["December", "March", "May"],
            "event_triggers": ["family emergency", "holiday remittance", "school fees", "medical bills"]},
    },
    "gig_worker": {
        "archetype": "gig_worker",
        "description": "Account takeover and payment fraud targeting gig economy workers",
        "demographics": {"age_range": (18, 45), "income_bracket": "low-to-middle", "banking_status": "banked, app-dependent",
            "primary_communities": ["Rideshare drivers (Uber, Lyft)", "Delivery drivers (DoorDash, Instacart)",
                "Freelancers (Fiverr, Upwork)", "Indian diaspora tech workers", "Tamil diaspora gig workers"]},
        "language_mix": {"en": 0.89, "hi": 0.06, "vi": 0.02, "es": 0.02, "yo": 0.01},
        "fraud_vectors": {"stolen": 0.18, "hacked": 0.13, "account takeover": 0.11, "ATO": 0.11, "PayPal": 0.10,
            "Venmo": 0.04, "CashApp": 0.04, "OTP": 0.04, "SIM swap": 0.03, "fake support": 0.03, "social engineering": 0.03, "unknown": 0.16},
        "instruments": ["CashApp", "Venmo", "Zelle", "PayPal", "Uber instant pay", "DoorDash direct deposit", "Bank debit card", "Prepaid card"],
        "transaction_patterns": {"amount_range_usd": (5, 3000), "median_amount_usd": 150,
            "peak_days": ["Sunday", "Monday"], "peak_hours": (20, 2), "typical_fee_pct": (1.0, 5.0)},
        "temporal": {"seasonal_peaks": ["January", "June", "November"],
            "event_triggers": ["platform policy change", "new driver onboarding", "instant pay feature launch", "SIM swap wave"]},
    },
    "unbanked": {
        "archetype": "unbanked",
        "description": "Prepaid card, payday loan, and kiosk fraud targeting unbanked populations",
        "demographics": {"age_range": (20, 60), "income_bracket": "low", "banking_status": "unbanked or underbanked",
            "primary_communities": ["Low-income urban populations", "Vietnamese diaspora", "Somali refugees",
                "Rural underserved communities", "Recently arrived immigrants"]},
        "language_mix": {"en": 0.84, "vi": 0.12, "yo": 0.02, "es": 0.01, "hi": 0.01},
        "fraud_vectors": {"predatory": 0.24, "prepaid": 0.16, "kiosk": 0.12, "advance fee": 0.08,
            "fake loan": 0.06, "load fee": 0.05, "payday loan": 0.04, "hawala": 0.03, "unknown": 0.22},
        "instruments": ["Prepaid Visa/Mastercard", "Green Dot card", "NetSpend", "Payday loan",
            "Check cashing service", "Bill pay kiosk", "Money order", "Cash (informal)"],
        "transaction_patterns": {"amount_range_usd": (10, 1500), "median_amount_usd": 200,
            "peak_days": ["Friday", "Monday"], "peak_hours": (9, 18), "typical_fee_pct": (3.0, 15.0)},
        "temporal": {"seasonal_peaks": ["January", "April", "August"],
            "event_triggers": ["tax refund season", "utility bill due", "rent due cycle", "predatory ad campaigns"]},
    },
    "itin": {
        "archetype": "itin",
        "description": "Identity theft, tax fraud, and synthetic identity targeting ITIN holders",
        "demographics": {"age_range": (22, 55), "income_bracket": "low-to-middle", "banking_status": "mixed",
            "primary_communities": ["Undocumented immigrants", "ITIN small business owners",
                "Indian H1B/visa holders", "Chinese immigrant entrepreneurs", "Korean small business owners", "Latin American immigrants"]},
        "language_mix": {"en": 0.96, "vi": 0.02, "ta": 0.01, "es": 0.01},
        "fraud_vectors": {"ITIN": 0.25, "EIN": 0.19, "identity theft": 0.13, "synthetic identity": 0.08,
            "tax return": 0.06, "immigration consultant": 0.05, "social security": 0.04, "mule": 0.03, "fake visa": 0.02, "unknown": 0.15},
        "instruments": ["ITIN tax filing", "EIN business registration", "Credit application",
            "Bank account opening", "Mortgage application", "Small business loan", "Fake immigration consultancy"],
        "transaction_patterns": {"amount_range_usd": (100, 50000), "median_amount_usd": 3000,
            "peak_days": ["Monday", "Tuesday", "Wednesday"], "peak_hours": (9, 17), "typical_fee_pct": (1.0, 10.0)},
        "temporal": {"seasonal_peaks": ["January", "February", "March", "April"],
            "event_triggers": ["tax filing season", "ITIN renewal deadline", "immigration policy change", "data breach exposure"]},
    },
}

print(f"Loaded {len(PROFILES)} archetype profiles:")
for name, p in PROFILES.items():
    print(f"  {name}: {len(p['fraud_vectors'])} vectors, {len(p['instruments'])} instruments, {len(p['language_mix'])} languages")

## 3. Seed Data Builder + Numerical Encoder + Categorical Sampler

In [ ]:
NUMERICAL_COLS = [
    "transaction_amount_usd", "fee_amount_usd", "sender_age", "hour_of_day",
    "day_of_week", "days_since_last_txn", "account_age_days", "txn_count_30d",
]
LABEL_COL = "is_fraud"


def build_seed_data(profile, n_samples=3000):
    """Generate seed training data (numericals + label only) from profile."""
    rng = np.random.default_rng(42)
    tp = profile["transaction_patterns"]
    demo = profile["demographics"]

    fraud_rate = 0.10
    is_fraud = rng.choice([0, 1], size=n_samples, p=[1 - fraud_rate, fraud_rate])
    fraud_mask = is_fraud == 1

    amt_lo, amt_hi = tp["amount_range_usd"]
    amounts = rng.lognormal(mean=np.log(tp["median_amount_usd"]), sigma=0.8, size=n_samples)
    amounts = np.clip(amounts, amt_lo, amt_hi * 1.5)
    amounts[fraud_mask] *= rng.uniform(1.2, 3.0, size=fraud_mask.sum())
    amounts = np.clip(amounts, amt_lo, amt_hi * 2)

    fee_lo, fee_hi = tp.get("typical_fee_pct", (1.0, 10.0))
    fees = amounts * rng.uniform(fee_lo, fee_hi, size=n_samples) / 100.0

    age_lo, age_hi = demo["age_range"]
    ages = rng.integers(age_lo, age_hi + 1, size=n_samples)
    hours = np.clip(rng.normal(loc=np.mean(tp.get("peak_hours", (9, 17))), scale=3.0, size=n_samples), 0, 23).astype(int)
    hours[fraud_mask] = rng.integers(0, 24, size=fraud_mask.sum())

    peak_day_map = {"Monday": 0, "Tuesday": 1, "Wednesday": 2, "Thursday": 3, "Friday": 4, "Saturday": 5, "Sunday": 6}
    weights = np.ones(7) * 0.1
    for d in tp.get("peak_days", ["Friday"]):
        weights[peak_day_map.get(d, 4)] = 0.25
    weights /= weights.sum()
    dow = rng.choice(7, size=n_samples, p=weights)

    days_since = np.clip(rng.exponential(scale=5.0, size=n_samples), 0, 90).astype(int)
    account_age = rng.integers(30, 2000, size=n_samples)
    account_age[fraud_mask] = rng.integers(1, 180, size=fraud_mask.sum())
    txn_count = rng.poisson(lam=8, size=n_samples)
    txn_count[fraud_mask] = rng.poisson(lam=20, size=fraud_mask.sum())

    return pd.DataFrame({
        "transaction_amount_usd": np.round(amounts, 2), "fee_amount_usd": np.round(fees, 2),
        "sender_age": ages, "hour_of_day": hours, "day_of_week": dow,
        "days_since_last_txn": days_since, "account_age_days": account_age, "txn_count_30d": txn_count,
        "is_fraud": is_fraud,
    })


class NumericalEncoder:
    """Standardize numerical columns for Gaussian diffusion."""
    def __init__(self, df, num_cols):
        self.num_cols = num_cols
        self.num_mean = df[num_cols].mean().values.astype(np.float32)
        self.num_std = df[num_cols].std().values.astype(np.float32)
        self.num_std[self.num_std < 1e-8] = 1.0

    def encode(self, df):
        X_num = torch.tensor((df[self.num_cols].values - self.num_mean) / self.num_std, dtype=torch.float32)
        y = torch.tensor(df[LABEL_COL].values, dtype=torch.long)
        return X_num, y

    def decode(self, X_num):
        return pd.DataFrame(X_num * self.num_std + self.num_mean, columns=self.num_cols)


def sample_categoricals(profile, n_samples):
    """Sample categoricals directly from profile distributions (no diffusion)."""
    rng = np.random.default_rng()
    vectors = list(profile["fraud_vectors"].keys())
    vp = np.array(list(profile["fraud_vectors"].values())); vp /= vp.sum()
    langs = list(profile["language_mix"].keys())
    lp = np.array(list(profile["language_mix"].values())); lp /= lp.sum()
    return pd.DataFrame({
        "fraud_vector": rng.choice(vectors, size=n_samples, p=vp),
        "language": rng.choice(langs, size=n_samples, p=lp),
        "instrument": rng.choice(profile["instruments"], size=n_samples),
    })


print("Seed builder, encoder, and categorical sampler defined")

In [ ]:
def train_tabddpm(df, encoder, epochs=700, lr=1e-3, batch_size=256, num_timesteps=500, device="cuda"):
    """Train Gaussian-only Tab-DDPM on numerical features."""
    X_num, y = encoder.encode(df)
    n_num = X_num.shape[1]
    num_label_classes = df[LABEL_COL].nunique()

    # No categorical classes -- pure Gaussian diffusion
    num_classes_array = np.array([0])

    denoise_fn = MLPDiffusion(
        d_in=n_num, num_classes=num_label_classes, is_y_cond=True,
        rtdl_params={"d_layers": [256, 256, 256], "dropout": 0.1}, dim_t=128,
    ).to(device)

    model = GaussianMultinomialDiffusion(
        num_classes=num_classes_array, num_numerical_features=n_num,
        denoise_fn=denoise_fn, num_timesteps=num_timesteps,
        gaussian_loss_type="mse", scheduler="cosine", device=torch.device(device),
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_num, y), batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(epochs):
        epoch_loss, n_batches = 0.0, 0
        for b_num, b_y in loader:
            b_num, b_y = b_num.to(device), b_y.to(device)
            loss_multi, loss_gauss = model.mixed_loss(b_num, {"y": b_y})
            loss = (loss_multi + loss_gauss).mean()
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item(); n_batches += 1
        if (epoch + 1) % 100 == 0 or epoch == 0:
            print(f"    Epoch {epoch+1:>4d}/{epochs} | loss={epoch_loss/max(n_batches,1):.4f}")

    return model


def generate_synthetic(model, encoder, archetype, profile, n_samples=5000, batch_size=1000, device="cuda"):
    """Sample numericals from diffusion, categoricals from profile, merge."""
    model.eval()
    y_dist = torch.tensor([0.90, 0.10])

    with torch.no_grad():
        x_gen, y_gen = model.sample_all(n_samples, batch_size, y_dist, ddim=False)

    x_np, y_np = x_gen.numpy(), y_gen.numpy()

    # Decode numericals from diffusion
    df_num = encoder.decode(x_np)

    # Sample categoricals from profile distributions (no diffusion)
    df_cat = sample_categoricals(profile, n_samples)

    df = pd.concat([df_num, df_cat], axis=1)
    df["is_fraud"] = y_np

    # Clamp values
    tp = profile["transaction_patterns"]
    amt_lo, amt_hi = tp["amount_range_usd"]
    df["transaction_amount_usd"] = df["transaction_amount_usd"].clip(amt_lo, amt_hi * 2).round(2)
    df["fee_amount_usd"] = df["fee_amount_usd"].clip(0, amt_hi).round(2)
    df["sender_age"] = df["sender_age"].clip(*profile["demographics"]["age_range"]).astype(int)
    df["hour_of_day"] = df["hour_of_day"].clip(0, 23).astype(int)
    df["day_of_week"] = df["day_of_week"].clip(0, 6).astype(int)
    df["days_since_last_txn"] = df["days_since_last_txn"].clip(0, 90).astype(int)
    df["account_age_days"] = df["account_age_days"].clip(1, 2000).astype(int)
    df["txn_count_30d"] = df["txn_count_30d"].clip(0, 100).astype(int)

    # Schema fields
    df.insert(0, "data_uuid", [str(uuid.uuid4()) for _ in range(len(df))])
    df.insert(1, "id", [f"synth_{uuid.uuid4().hex[:12]}" for _ in range(len(df))])
    df.insert(2, "archetype", archetype)
    df["fraud_vector_hint"] = df["fraud_vector"]
    df["detected_language_hints"] = df["language"].apply(lambda x: [x])
    df["narrative_text"] = ""  # placeholder for Adaptive Data
    base_date = datetime(2024, 1, 1, tzinfo=timezone.utc)
    df["record_timestamp"] = [
        (base_date + timedelta(days=int(np.random.randint(0, 365)),
         hours=int(row["hour_of_day"]), minutes=int(np.random.randint(0, 60)))).isoformat()
        for _, row in df.iterrows()]
    df["source"] = "tabddpm_synthetic"
    day_names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    df["day_of_week_name"] = df["day_of_week"].map(lambda x: day_names[x])

    # Reorder columns
    universal = ["data_uuid", "id", "archetype", "source", "narrative_text",
                 "detected_language_hints", "fraud_vector_hint", "record_timestamp"]
    txn = ["transaction_amount_usd", "fee_amount_usd", "sender_age", "hour_of_day",
           "day_of_week", "day_of_week_name", "days_since_last_txn", "account_age_days",
           "txn_count_30d", "fraud_vector", "language", "instrument", "is_fraud"]
    return df[universal + txn]


print("Hybrid train + generate functions defined")

## 4. Run: Hybrid generation (700 epochs, Gaussian diffusion + profile sampling)

In [ ]:
%%time

EPOCHS = 700
SEED_SIZE = 3000
N_SAMPLES = 5000

all_dfs = {}
all_stats = {}

for arch_name, profile in PROFILES.items():
    print(f"\n{'='*60}")
    print(f"  {arch_name.upper()}: {profile['description']}")
    print(f"{'='*60}")

    # Build seed data (numericals only)
    print(f"  [1/3] Building seed data ({SEED_SIZE} rows)...")
    df_seed = build_seed_data(profile, n_samples=SEED_SIZE)
    print(f"    Fraud rate: {df_seed['is_fraud'].mean():.1%}")

    # Encode numericals only
    encoder = NumericalEncoder(df_seed, NUMERICAL_COLS)

    # Train Gaussian-only diffusion
    print(f"  [2/3] Training Gaussian diffusion ({EPOCHS} epochs, 500 timesteps, {DEVICE})...")
    model = train_tabddpm(df_seed, encoder, epochs=EPOCHS, num_timesteps=500, device=DEVICE)

    # Generate (numericals from diffusion, categoricals from profile)
    print(f"  [3/3] Generating {N_SAMPLES} synthetic records...")
    df_synth = generate_synthetic(model, encoder, arch_name, profile, n_samples=N_SAMPLES, device=DEVICE)

    # Save
    parquet_path = f"/content/output/{arch_name}/transactions.parquet"
    csv_path = f"/content/output/{arch_name}/transactions_{arch_name}.csv"
    df_synth.to_parquet(parquet_path, index=False)
    df_synth.to_csv(csv_path, index=False)
    all_dfs[arch_name] = df_synth

    stats = {
        "records": len(df_synth),
        "fraud_rate": f"{df_synth['is_fraud'].mean():.1%}",
        "amt_median": f"${df_synth['transaction_amount_usd'].median():.0f}",
        "vectors": df_synth["fraud_vector"].nunique(),
        "languages": df_synth["language"].nunique(),
        "instruments": df_synth["instrument"].nunique(),
    }
    all_stats[arch_name] = stats
    print(f"  -> Saved: {parquet_path}")
    print(f"  -> Stats: {stats}")

    # Free GPU memory
    del model
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n{'='*60}")
print("ALL DONE")
print(f"{'='*60}")

## 5. Verify: Distribution alignment vs profile_configs

In [ ]:
for arch_name in PROFILES:
    df = all_dfs[arch_name]
    profile = PROFILES[arch_name]
    print(f"\n{'='*60}")
    print(f"  {arch_name.upper()} â€” Distribution Verification")
    print(f"{'='*60}")

    # Fraud vector distribution: expected vs actual
    print("\n  FRAUD VECTORS (expected -> actual):")
    actual_vec = df["fraud_vector"].value_counts(normalize=True).to_dict()
    for vec, expected_pct in sorted(profile["fraud_vectors"].items(), key=lambda x: -x[1]):
        actual_pct = actual_vec.get(vec, 0)
        drift = abs(actual_pct - expected_pct)
        flag = " <-- DRIFT" if drift > 0.10 else ""
        print(f"    {vec:<25} expected={expected_pct:.0%}  actual={actual_pct:.0%}{flag}")

    # Language distribution
    print("\n  LANGUAGES (expected -> actual):")
    actual_lang = df["language"].value_counts(normalize=True).to_dict()
    for lang, expected_pct in sorted(profile["language_mix"].items(), key=lambda x: -x[1]):
        actual_pct = actual_lang.get(lang, 0)
        drift = abs(actual_pct - expected_pct)
        flag = " <-- DRIFT" if drift > 0.10 else ""
        print(f"    {lang:<10} expected={expected_pct:.0%}  actual={actual_pct:.0%}{flag}")

    # Amount stats
    tp = profile["transaction_patterns"]
    print(f"\n  AMOUNTS:")
    print(f"    Expected range:  ${tp['amount_range_usd'][0]} - ${tp['amount_range_usd'][1]}")
    print(f"    Expected median: ${tp['median_amount_usd']}")
    print(f"    Actual range:    ${df['transaction_amount_usd'].min():.0f} - ${df['transaction_amount_usd'].max():.0f}")
    print(f"    Actual median:   ${df['transaction_amount_usd'].median():.0f}")

    # Instrument count
    print(f"\n  INSTRUMENTS: expected={len(profile['instruments'])} actual={df['instrument'].nunique()}")
    print(f"    {df['instrument'].value_counts().to_dict()}")

## 6. Download outputs

In [ ]:
# Zip all outputs and download
import shutil
shutil.make_archive("/content/tabddpm_output", "zip", "/content/output")

from google.colab import files
files.download("/content/tabddpm_output.zip")
print("Download started â€” unzip into datasets/ folder locally")

## 7. Quick preview

In [ ]:
# Preview first 5 rows of each archetype
for arch_name, df in all_dfs.items():
    print(f"\n--- {arch_name.upper()} (first 5 rows) ---")
    display(df[["data_uuid", "archetype", "transaction_amount_usd", "fraud_vector", "language", "instrument", "is_fraud"]].head())